# Self-Normalized Phase-Space Adaptive Moving Average (PSAMA)

This demo notebook implements and evaluates the **Self-Normalized Phase-Space Adaptive Moving Average (PSAMA)** method. By computing rolling median absolute deviation (MAD) normalized gradient volatility, PSAMA dynamically scales moving average window lengths to balance responsiveness during high-volatility regime shifts and smoothing during stochastic noise.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'loguru==0.7.3')

In [ ]:
import json
import numpy as np
from pathlib import Path
from loguru import logger
import matplotlib.pyplot as plt

import sys
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-2/experiment-1/demo/mini_demo_data.json"
import urllib.request

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.info(f"Failed to load from GitHub URL ({e}), falling back to local file.")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: 
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os
data = load_data()
logger.info(f"Loaded dataset with {len(data['datasets'])} groups.")

## Method Implementation & Evaluation

We compute rolling median absolute deviation (MAD), naive persistence, static MA(3), unnormalized PSAMA, and self-normalized PSAMA across the input sequences.

In [ ]:
def rolling_mad(series, window=5):
    mad = np.zeros_like(series)
    for i in range(len(series)):
        start = max(0, i - window + 1)
        chunk = series[start:i+1]
        med = np.median(chunk)
        mad[i] = np.median(np.abs(chunk - med)) + 1e-8
    return mad

all_datasets = []
for ds in data["datasets"]:
    ds_name = ds["dataset"]
    logger.info(f"Processing dataset group: {ds_name}")
    
    examples_out = []
    for ex in ds["examples"]:
        inp = json.loads(ex["input"])
        out = json.loads(ex["output"])
        
        pred_naive = [inp[0]] + inp[:-1]
        
        pred_static_ma = []
        for i in range(len(inp)):
            start = max(0, i - 2)
            pred_static_ma.append(float(np.mean(inp[start:i+1])))
            
        pred_unnorm_psama = []
        for i in range(len(inp)):
            if i == 0:
                pred_unnorm_psama.append(inp[0])
                continue
            grad = abs(inp[i] - inp[i-1])
            w = int(np.clip(round(3 / (1.0 + grad * 5.0)), 1, 5))
            start = max(0, i - w + 1)
            pred_unnorm_psama.append(float(np.mean(inp[start:i+1])))
            
        mad_series = rolling_mad(np.array(inp), window=5)
        pred_self_norm_psama = []
        for i in range(len(inp)):
            if i == 0:
                pred_self_norm_psama.append(inp[0])
                continue
            grad = abs(inp[i] - inp[i-1])
            norm_grad = grad / mad_series[i]
            w = int(np.clip(round(3 / (1.0 + norm_grad * 5.0)), 1, 5))
            start = max(0, i - w + 1)
            pred_self_norm_psama.append(float(np.mean(inp[start:i+1])))
            
        example_entry = {
            "input": ex["input"],
            "output": ex["output"],
            "metadata_id": str(ex["metadata_id"]),
            "metadata_process_type": str(ex["metadata_process_type"]),
            "metadata_noise_level": str(ex["metadata_noise_level"]),
            "predict_naive_persistence": json.dumps(pred_naive),
            "predict_static_ma3": json.dumps(pred_static_ma),
            "predict_unnormalized_psama": json.dumps(pred_unnorm_psama),
            "predict_self_normalized_psama": json.dumps(pred_self_norm_psama)
        }
        examples_out.append(example_entry)
        
    all_datasets.append({
        "dataset": ds_name,
        "examples": examples_out
    })

output_data = {
    "metadata": {
        "experiment": "Self-Normalized Phase-Space Adaptive Moving Average"
    },
    "datasets": all_datasets
}

out_path = Path("full_method_out.json")
out_path.write_text(json.dumps(output_data, indent=2))
logger.info(f"Successfully saved experiment results to {out_path}")

## Results Visualization

Let's visualize the input series alongside the predictions from Naive Persistence, Static MA(3), Unnormalized PSAMA, and Self-Normalized PSAMA for the first example.

In [ ]:
ex0 = output_data["datasets"][0]["examples"][0]
inp_vals = json.loads(ex0["input"])
naive_vals = json.loads(ex0["predict_naive_persistence"])
ma3_vals = json.loads(ex0["predict_static_ma3"])
unnorm_vals = json.loads(ex0["predict_unnormalized_psama"])
selfnorm_vals = json.loads(ex0["predict_self_normalized_psama"])

plt.figure(figsize=(10, 5))
plt.plot(inp_vals, label="Input (Noisy Series)", color="black", alpha=0.6, linestyle="--")
plt.plot(naive_vals, label="Naive Persistence", color="red", alpha=0.7)
plt.plot(ma3_vals, label="Static MA(3)", color="blue", alpha=0.7)
plt.plot(unnorm_vals, label="Unnormalized PSAMA", color="orange", alpha=0.7)
plt.plot(selfnorm_vals, label="Self-Normalized PSAMA", color="green", linewidth=2)
plt.legend()
plt.title("PSAMA Method Comparison on Time Series Example")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.grid(True, alpha=0.3)
plt.show()